# Taller de Modelado de Riesgo Crediticio (HELOC)
Autor: _[Tu Nombre Aquí]_  
Fecha: _[Completar]_  

Este cuaderno contiene las instrucciones para completar tareas de regresión y clasificación sobre el dataset **Home‑Equity Line of Credit (HELOC)**. Sigue cada sección, agrega tus celdas de código donde se indique **TODO**, y responde las preguntas solicitadas.

| Aspecto | Detalle |
|---------|---------|
| **Origen** | Publicado por **FICO®** como parte de su “Explainable Machine Learning Challenge” (2018); disponible en Kaggle y en el repositorio oficial de la competencia. |
| **Observaciones** | **10 459** solicitudes de línea de crédito hipotecaria (clientes únicos). |
| **Variables** | **24 columnas** en total: 1 objetivo (`RiskPerformance`) + 23 predictoras numéricas y categóricas codificadas como enteros. |
| **Variable objetivo** | `RiskPerformance` &rarr; “Good” (0) ó “Bad” (1). “Bad” indica que el cliente incurrió en default, morosidad grave o cierre por cobranza. |
| **Variables predictoras clave** | `ExternalRiskEstimate`, `MSinceMostRecentDelq`, `NumTotalTrades`, `PercentTradesNeverDelq`, `NetFractionRevolvingBurden`, entre otras. Representan historiales de crédito, antigüedad de líneas, morosidades y utilización de cupo. |
| **Codificación de faltantes** | El valor **–8** indica “no disponible” (missing). No existen `NaN` nativos; se debe convertir –8 → `NaN` antes de imputar o escalar. |
| **Balance de clases** | ~ **74 %** “Good” vs. **26 %** “Bad” (dataset moderadamente desbalanceado). |
| **Posibles tareas** | 1) **Clasificación** del riesgo (`RiskPerformance`).<br>2) **Regresión** del score de riesgo (`ExternalRiskEstimate`) o métricas de carga financiera (`NetFraction*`). |
| **Licencia / uso** | Datos anonimizados, liberados exclusivamente con fines académicos y de investigación; no incluyen información personal identificable. |

## Codificación de Faltantes

| Value                | Meaning                                                            |
| -------------------- | ------------------------------------------------------------------ |
| `-9`                 | **No Bureau Record or No Matching Trade** (missing / not reported) |
| `-8`                 | No Usable/Valid Trades                                             |
| `-7`                 | Condition not met                                                  |
| other numeric values | Valid feature values                                               |


## 1. Carga de librerías y datos
Ejecuta la celda siguiente para cargar el CSV que ya descargaste (`heloc_dataset_v1.csv`). Si lo tienes en otra ruta, ajusta el _path_.

In [ ]:
import pandas as pd
import numpy as np

# Cargar datos
df = pd.read_csv('heloc_dataset_v1.csv')

# Vista preliminar
df.head()

# 2. Exploración de los datos

In [ ]:
# Información general
df.info()

In [ ]:
# Descripción de los datos
df.describe()

# 3. Preprocesamiento de datos

## 📖 Guía Rápida: Criterios para la Limpieza y Transformación de Datos

Antes de aplicar cualquier algoritmo, las decisiones que tomamos sobre los datos faltantes, las escalas y los valores extremos impactan directamente en el rendimiento del modelo. Aquí tienes una guía de referencia para saber cuándo usar cada técnica:

### 1. Imputación de Valores Faltantes (Nulos)


La elección de la medida de tendencia central depende de la forma de la distribución de tus datos:

* **Media (Promedio):** * **Cuándo usarla:** Cuando la variable tiene una distribución normal (simétrica) o gaussiana.
    * **Precaución:** Es extremadamente sensible a los valores atípicos (outliers). Si hay un valor extremo, el promedio se arrastrará hacia él, distorsionando la imputación.
* **Mediana:** * **Cuándo usarla:** Cuando la distribución es asimétrica (sesgada) o cuando hay presencia de valores atípicos claros. Es la medida más robusta para datos financieros (como salarios, deudas o créditos).
* **Moda:** * **Cuándo usarla:** Exclusivamente para variables categóricas (ej. estado civil, nivel educativo) o variables numéricas discretas con muy poca variabilidad (ej. número de hijos, donde la mayoría tiene 0 o 1).

---

### 2. Escalamiento de Datos
Muchos algoritmos de Machine Learning (como PCA, KNN o SVM) calculan distancias entre puntos. Si una variable está en miles (ej. ingresos) y otra en decimales (ej. tasas de interés), el modelo le dará más peso a la primera solo por su magnitud.

| Técnica | ¿Qué hace? | ¿Cuándo usarla? |
| :--- | :--- | :--- |
| **Estandarización (StandardScaler)** | Centra los datos en una media de 0 y una desviación estándar de 1 (Z-score). No acota los datos a un rango específico. | Es el estándar para algoritmos que asumen normalidad o calculan varianzas, **como el PCA**, Regresión Lineal o Regresión Logística. Es menos sensible a outliers que la normalización. |
| **Normalización (MinMaxScaler)** | Comprime todos los datos a un rango fijo, generalmente entre 0 y 1. | Útil cuando no asumimos ninguna distribución en particular, en Redes Neuronales, procesamiento de imágenes, o cuando los límites exactos (0 a 1) son importantes. **Precaución:** Muy sensible a los outliers, ya que un valor extremo aplastará al resto de los datos en un rango diminuto. |

---

### 3. El Dilema de los Valores Atípicos (Outliers)

Un *outlier* no siempre es un error; muchas veces es la información más valiosa del dataset (por ejemplo, en detección de fraudes o en la predicción de quiebras crediticias).

* **Cuándo NO imputar ni eliminar (Mantenerlos):**
    * Cuando representan eventos reales y críticos que el modelo necesita aprender (ej. clientes de muy alto riesgo en nuestro dataset HELOC).
    * Cuando vas a usar modelos basados en árboles (Random Forest, XGBoost), ya que estos algoritmos son robustos y manejan los valores extremos naturalmente mediante particiones.
* **Cuándo tratar, eliminar o imputar (Capping/Winsorización):**
    * Cuando sabes con certeza que es un error de recolección de datos (ej. una edad de 150 años).
    * Cuando vas a usar modelos muy sensibles a la magnitud, como **PCA**, K-Means o Regresión Lineal, donde un solo valor extremo puede desviar completamente el hiperplano o los centroides. En estos casos, se suele limitar el valor a los "bigotes" del boxplot (técnica de recorte o *capping*).